In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, struct, to_json, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

In [25]:
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()

In [26]:
user_df = spark.read.csv("data/user_table.csv", header= True, inferSchema= True)


In [27]:
# 2. Read Streaming Data from Kafka
tx_schema = StructType({
    StructField("tx_id", IntegerType(), True),
     StructField("userId", IntegerType(), True),
     StructField("amount", DoubleType(), True),
     StructField("timestamp", DoubleType(), True)
})

kafka_stream = spark.readStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("subscribe", "fraud-detection") \
 .load()

In [28]:
# 3. Parse and Filter Data
parsed_stream = kafka_stream.select(
    from_json(
        col("value").cast("string"), 
        tx_schema
    ).alias("tx")
).select("tx.*")

In [29]:
# Fraud Logic: Amount > 10,000
fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [ ]:
# 4. Enrich Stream with User Details (Stream-Static Join)
enriched_fraud = fraud_stream.join(user_df, "userId")
# query = enriched_fraud.writeStream \           for printing data on console
#     .outputMode("append") \
#     .format("console") \
#     .start()

# query.awaitTermination()

26/06/19 05:03:40 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-89c07361-8b17-4906-9234-4e2953f4a454. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/19 05:03:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 05:03:40 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+---------+-----+------+----+-----+-----+--------------+
|userId|timestamp|tx_id|amount|name|email|phone|account_status|
+------+---------+-----+------+----+-----+-----+--------------+
+------+---------+-----+------+----+-----+-----+--------------+



In [34]:
# 5. Format for output Kafka topic
output_stream = enriched_fraud \
 .withColumn("value", to_json(struct("*")).cast("string")) \
 .select("value")

In [39]:
# 6. Write Stream to 'fraud-notification' Topic
query = output_stream.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification") \
 .option("checkpointLocation", "/tmp/checkpoints") \
 .start()
query.awaitTermination()

26/06/19 05:01:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 05:01:33 WARN StreamingQueryManager: Stopping existing streaming query [id=3840f2ac-e13d-4950-ac09-2114f4ffeaa5, runId=9a2a03c3-0914-4e27-92ce-4370d8dba392], as a new run is being started.


AttributeError: 'StreamingQuery' object has no attribute 'values'

26/06/19 05:01:33 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
